# Laborator 6 – K-Nearest Neighbors pe setul de date Iris

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import warnings
warnings.filterwarnings('ignore')

# 1. Explorarea setului de date
iris = load_iris(as_frame=True)
print(f"Nr. exemple: {iris.data.shape[0]}, Nr. caracteristici: {iris.data.shape[1]}")
print(f"Atribute: {iris.feature_names}")
print(f"Clase: {iris.target_names.tolist()}")
iris.frame.head()

In [ ]:
# 2. Impartire train/test 80/20
X = iris.data.values
y = iris.target.values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train labels: {X_train.shape[0]}, Test labels: {X_test.shape[0]}")

In [ ]:
# 3. Preprocesare – StandardScaler
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print("Primele 3 exemple INAINTE de scalare:")
print(X_test[:3].round(4))
print("Primele 3 exemple DUPA scalare:")
print(X_test_sc[:3].round(4))

In [ ]:
# 4. Model KNN cu k=3
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train_sc, y_train)
acc = accuracy_score(y_test, knn.predict(X_test_sc))
print(f"Acuratete KNN (k=3): {acc:.4f} ({acc*100:.2f}%)")

In [ ]:
# 5. Impact k (1-15)
accuracies = []
for k in range(1, 16):
    m = KNeighborsClassifier(n_neighbors=k)
    m.fit(X_train_sc, y_train)
    accuracies.append(accuracy_score(y_test, m.predict(X_test_sc)))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, 16), accuracies, marker='o', color='steelblue')
ax.set_xlabel('k')
ax.set_ylabel('Acuratete')
ax.set_title('Acuratete KNN in functie de k')
ax.set_xticks(range(1, 16))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('lab6_k_accuracy.png', dpi=80)
plt.show()

best_k = np.argmax(accuracies) + 1
print(f"Cel mai bun k: {best_k} cu acuratete {accuracies[best_k-1]:.4f}")

In [ ]:
# 6. Evaluare model
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_train_sc, y_train)
y_pred = knn_best.predict(X_test_sc)

print("Matrice de confuzie:")
print(confusion_matrix(y_test, y_pred))
print()
print("Raport clasificare:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

In [ ]:
# 7. Vizualizare scatter 2D (petal length vs petal width)
fig, ax = plt.subplots(figsize=(7, 5))
colors = ['red','green','blue']
for i, cls in enumerate(iris.target_names):
    mask = y == i
    ax.scatter(X[mask, 2], X[mask, 3], c=colors[i], label=cls, alpha=0.7)
ax.set_xlabel('Lungime petala (cm)')
ax.set_ylabel('Latime petala (cm)')
ax.set_title('Iris – Lungime vs Latime petala')
ax.legend()
plt.tight_layout()
plt.savefig('lab6_scatter.png', dpi=80)
plt.show()

In [ ]:
# Predictie pentru o floare noua (demo fara input interactiv)
sample = np.array([[5.1, 3.5, 1.4, 0.2]])   # Iris setosa typical values
sample_sc = scaler.transform(sample)
pred_class = iris.target_names[knn_best.predict(sample_sc)[0]]
print(f"Floare noua: {sample[0].tolist()}")
print(f"Specia prezisa: {pred_class}")

# Versiune cu input() - decomentati pentru rulare interactiva
# vals = [float(input(f'Introduceti {f}: ')) for f in iris.feature_names]
# pred = iris.target_names[knn_best.predict(scaler.transform([vals]))[0]]
# print(f'Specia prezisa: {pred}')
